# Глубинное обучение для текстовых данных, ФКН ВШЭ
## Домашнее задание 3: Уменьшение размеров модели
### Оценивание и штрафы

Максимально допустимая оценка за работу — __16 баллов__.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом для пользоваться запрещается даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

__Мягкий дедлайн 22.11.26 23:59__ \
__Жесткий дедлайн 24.11.26 23:59__

### О задании

В этом задании вам предстоит научиться решать задачу Named Entity Recognition (NER) на самом популярном датасете – [CoNLL-2003](https://paperswithcode.com/dataset/conll-2003). В вашем распоряжении будет предобученный BERT, который вам необходимо уменьшить с минимальными потерями в качестве до размера 20М параметров. Для этого вы самостоятельно реализуете факторизацию эмбеддингов, дистилляцию, шеринг параметров и так далее.

В этом задании вам придется проводить довольно много экспериментов, поэтому мы рекомендуем не писать весь код в тетрадке, а завести разные файлы для отдельных логических блоков и скомпоновать все в виде проекта. Это позволит вашему ноутбуку не разрастаться и сильно облегчит задачу и вам, и проверяющим. Так же постарайтесь логгировать все ваши эксперименты на wandb или других платформах, чтобы ничего не потерялось.

В этом задании __запрещается использовать агентов для написания кода__, но разрешается для помощи в чтении статей и осознании методов.

### Оценивание
Оценка за это домашнее задание будет формироваться из оценки за __задания__ и за __отчет__, в котором от вас требуется написать о проделанной работе. За отчет можно получить до 2-х баллов, однако в случае отсутствия отчета баллы за соответствующие задания не будут ставиться. Задания делятся на две части: _номерные_ и _на выбор_. За _номерные_ можно получить в сумме 6 баллов, за задания _на выбор_ можно получить до 8. То есть за все дз можно получить 16 баллов (но не радуйтесь рано, это не так просто). Все, что вы наберете свыше 10, будет считаться бонусами.

### О датасете

Named Entity Recognition – это задача классификации токенов по классам сущностей. В CoNLL-2003 для именования сущностей используется маркировка **BIO** (Beggining, Inside, Outside), в которой метки означают следующее:

- *B-{метка}* – начало сущности *{метка}*
- *I-{метка}* – продолжнение сущности *{метка}*
- *O* – не сущность

Существуют так же и другие способы маркировки, например, BILUO. Почитать о них можно [тут](https://en.wikipedia.org/wiki/Inside–outside–beginning_(tagging)) и [тут](https://www.youtube.com/watch?v=dQw4w9WgXcQ).

Всего в датасете есть 9 разных меток.
- O – слову не соответствует ни одна сущность.
- B-PER/I-PER – слово или набор слов соответстует определенному _человеку_.
- B-ORG/I-ORG – слово или набор слов соответстует определенной _организации_.
- B-LOC/I-LOC – слово или набор слов соответстует определенной _локации_.
- B-MISC/I-MISC – слово или набор слов соответстует сущности, которая не относится ни к одной из предыдущих. Например, национальность, произведение искусства, мероприятие и т.д.

Приступим!

Начнем с загрузки и предобработки датасета.

In [1]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path('homeworks').resolve()))
os.environ['WANDB_MODE'] = 'offline'
os.environ.setdefault('WANDB_PROJECT', 'BERT_EXP')

from hw3_distillation.data import load_data
from hw3_distillation.models import BertForNER, factorize, load_teacher, make_student
from hw3_distillation.train import RUNS_DIR, train

dataset, tokenizer = load_data()
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

### Предобработка

На протяжении всего домашнего задания мы будем использовать _cased_ версию BERT, то есть токенизатор будет учитывать регистр слов. Для задачи NER регистр важен, так как имена и названия организаций или предметов искусства часто пишутся с большой буквы, и будет глупо прятать от модели такую информацию.

При токенизации слова могут разделиться на несколько токенов (как слово `Fischler` из примера ниже), из-за чего появится несоответствие между числом токенов и меток. Это несоответствие нам придется устранить вручную.

__Задание 1 (1 балл).__ Токенизируйте весь датасет и для каждого текста выравните токены с метками так, чтобы каждому токену соответствовала одна метка. При этом важно сохранить нотацию BIO. И не забудьте про специальные токены! Должно получиться что-то такое:

### Метрика

Для оценки качества NER обычно используют F1 меру с микро-усреднением. Мы загрузим ее из библиотеки `seqeval`. Функция `f1_score` принимает два 2d списка с правильными и предсказанными метками, записаными текстом, и возвращает для них значение F1. Вы можете использовать ее с параметрами по умолчанию.

Особенность подсчета F1 для NER заключается в том, что в некоторых ситуациях неправильные ответы могут засчитываться как правильные. Например, если модель предсказала `['I-PER', 'I-PER']`, то мы можем догадаться, что на самом деле должно быть `['B-PER', 'I-PER']`, так как сущность не может начинаться с `I-`. Функция `f1_score` учитывает это и поэтому работает только с текстовыми представлениями меток.

### Модель

В качестве базовой модели мы возьмем `bert-base-cased`. Как вы понимаете, он не обучался на задачу NER. Поэтому прежде чем приступать к уменьшению размера BERT, его необходимо дообучить.

__Задание 2 (1 балл)__ Дообучите `bert-base-cased` на нашем датасете с помощью обычного fine-tuning. У вас должно получиться хотя бы 0.9 F1 на тестовой выборке. Заметьте, что чем выше качество большой модели, тем лучше будет работать дистиллированный ученик. Для обучения можно использовать `Trainer` из Hugging Face.

In [2]:
train(
    load_teacher(),
    dataset,
    tokenizer,
    'teacher'
)

{'model': 'teacher',
 'params': 107726601,
 'val_f1': 0.9397068620919389,
 'test_f1': 0.8949099722991688}

In [ ]:
train(
    load_teacher(),
    dataset,
    tokenizer,
    'teacher_lr2',
    learning_rate=2e-5
)

In [ ]:
train(
    load_teacher(),
    dataset,
    tokenizer,
    'teacher_lr5',
    learning_rate=5e-5
)

### Учитель, batch size 16

In [3]:
train(
    load_teacher(),
    dataset,
    tokenizer,
    'teacher_b16',
    num_train_epochs=6,
    learning_rate=2e-5,
    per_device_train_batch_size=16
)

{'model': 'teacher_b16',
 'params': 107726601,
 'val_f1': 0.9447965310206805,
 'test_f1': 0.9033883579496091}

### Факторизация матрицы эмбеддингов

Можно заметить, что на данный момент матрица эмбеддингов занимает $V \cdot H = 28996 \cdot 768 = 22.268.928$ параметров. Это aж пятая часть от всей модели! Давайте попробуем что-то с этим сделать. В модели [ALBERT](https://arxiv.org/pdf/1909.11942.pdf) предлагается факторизовать матрицу эмбеддингов в произведение двух небольших матриц. Таким образом, параметры эмбеддингов будут содержать $V \cdot E + E \cdot H$ элементов, что гораздо меньше $V \cdot H$, если $H \gg E$. Авторы выбирают $E = 128$, однако ничего не мешает нам взять любое другое значение. Например, выбрав $H = 64$, мы уменьшим число параметров примерно на 20М.

__Задание 3 (1 балл).__ Напишите класс-обертку над слоем эмбеддингов, который реализует факторизацию на две матрицы, и дообучите факторизованную модель. Заметьте, обе матрицы можно инициализировать с помощью SVD разложения (а лучше [FWSVD](https://arxiv.org/pdf/2207.00112)!), чтобы начальное приближение было хорошим. Это сэкономит очень много времени на дообучении. С рангом разложения $H = 64$ у вас должно получиться F1 больше 0.87.

In [4]:
teacher_path = RUNS_DIR / 'teacher_b16' / 'best'
train(
    factorize(teacher_path),
    dataset,
    tokenizer,
    'svd64'
)

{'model': 'svd64',
 'params': 87362569,
 'val_f1': 0.9165556295802797,
 'test_f1': 0.8756080611535787}

### Дистилляция знаний

Дистилляция знаний – это парадигма обучения, в которой знания модели-учителя дистиллируются в модель-ученика. Учеником может быть произвольная модель меньшего размера, решающая ту же задачу, однако обычно ученик имеет ту же архитектуру, что и учитель. При дистилляции используются два функционала ошибки:

1. Стандартная кросс-энтропия.
1. Функция, задающая расстояние между распределениями предсказаний учителя и ученика. Чаще всего используют KL-дивергенцию.

Для того, чтобы распределение предсказаний учителя не было вырожденным, к softmax добавляют температуру больше 1, например, 2 или 5.   
__Важно:__ при делении логитов на температуру значения градиентов уменьшаются в $\tau^2$ раз (проверьте это!). Поэтому для возвращения их в изначальный масштаб ошибку надо домножить на $\tau^2$. Подробнее об этом можно почитать в разделе 2.1 [оригинальной статьи](https://arxiv.org/pdf/1503.02531).

<img src="https://intellabs.github.io/distiller/imgs/knowledge_distillation.png" width="800">

__Задание 4 (3 балла).__ Реализуйте метод дистилляции знаний, изображенный на картинке. Для подсчета ошибки между предсказаниями ученика и учителя используйте KL-дивергенцию [`nn.KLDivLoss(reduction="batchmean")`](https://pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html) (обратите внимание на вормат ее входов). Для получения итоговой ошибки суммируйте мягкую ошибку с жесткой.   
В качестве учителя используйте дообученный BERT из задания 2. В качестве ученика возьмите необученную модель с размером __не больше 20M__ параметров. Вы можете использовать факторизацию матрицы эмбеддингов для уменьшения числа параметров. Если вы все сделали правильно, то на тестовой выборке вы должны получить значение F1 не меньше 0.7. Вам должно хватить примерно 20к итераций обучения для этого. Если у вас что-то не получается, то можно ориентироваться на статью про [DistilBERT](https://arxiv.org/abs/1910.01108) и на [эту статью](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT).

__Важно:__
* Не забывайте добавлять _warmup_ при обучении ученика.
* Не забывайте переводить учителя в режим _eval_.

In [ ]:
student_args = dict(
    teacher_checkpoint=teacher_path,
    temperature=5,
    max_steps=20000,
    learning_rate=3e-4,
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=1000,
    save_steps=1000
)
train(
    make_student(tokenizer),
    dataset,
    tokenizer,
    'student_t5',
    **student_args
)

{'model': 'student_t5',
 'params': 19807753,
 'val_f1': 0.8077985105848883,
 'test_f1': 0.741216870652461}

In [ ]:
train(
    make_student(tokenizer),
    dataset,
    tokenizer,
    'student',
    **dict(student_args, temperature=2.0)
)

# Задания на выбор

Как вы понимаете, есть еще довольно много разных способов уменьшить обученную модель. В этой секции вам предлагается реализовать разные техники на выбор. За каждую из них можно получить разное количество балов в зависимости от сложности. Успешность реализации будет оцениваться как по коду, так и по качеству на тестовой выборке. Все баллы за это дз, которые вы наберете сверх 10, будут считаться бонусными.   
В задании 4 вы обучали модель с ограничением числа параметров в 20М. При реализации техник из этой секции придерживайтесь такого же ограничения. Это позволит честно сравнивать методы между собой и делать правильные выводы. Напишите в отчете обо всем, что вы попробовали.

* __Шеринг весов (1 балл).__ В модификации BERT [ALBERT](https://arxiv.org/pdf/1909.11942) помимо факторизации эмбеддингов предлагается шерить веса между слоями. То есть разные слои используют одни и те же веса. Такая техника эвивалентна применению одного и того же слоя несколько раз. Она позволяет в несколько раз уменьшить число параметров и не сильно потерять в качестве.
* __Шеринг весов++ (0.5 балл).__ Вместо того, чтобы просто применять одни и те же слои, можно дообучить LoRA для каждого слоя, чтобы слои могли специализироваться. Ровно так делают в [этой статье](https://arxiv.org/pdf/2410.20672) от DeepMind.
* __Приближение промежуточных слоев (1.5 балла).__ Мы обсуждали, что помимо приближения выходов модели ученика к выходам модели учителя, можно приближать выходы промежуточных слоев. В [этой работе](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT) подробно написано, как это можно сделать. Еще __+1 балл__ за реализацию метода из [MiniLMv2](https://arxiv.org/pdf/2012.15828).
* __Прунинг (3 балла).__ В методе [SparseGPT](https://arxiv.org/abs/2301.00774) предлагается подход, удаляющий веса модели один раз после обучения. При этом оказывается возможным удалить до половины всех весов без потери в качестве. Математика, стоящаяя за техникой, довольно сложная, однако общий подход простой – будем удалять веса в каждом слое по отдельности, при удалении части весов слоя, остальные веса будут перенастраиваться так, чтобы общий выход слоя не изменился. В качестве бейзлайна реализуйте более простой метод [Wanda](https://arxiv.org/pdf/2306.11695).
* __Инициализация ученика прунингом (2 балла).__ Обучение ученика с нуля занимает достаточно много времени. Вместо этого, ученика можно инициализоровать с помощью прунинга учителя, а затем дообучить с помощью дистилляции. Впервые так сделали в [Minitron](https://arxiv.org/pdf/2408.11796), а после в Ministral 3, Llama 3.2, Falcon3 и т.д. Важная проблема этого подхода заключается в том, что слои могут пруниться неравномерно, из-за чего при использовании модели будет сложнее распределять нагрузку. Но если задача – просто уменьшить размер, то это отличный способ.



### Шеринг весов

In [6]:
train(
    make_student(tokenizer, shared=True),
    dataset,
    tokenizer,
    'shared',
    **student_args
)

{'model': 'shared',
 'params': 7198217,
 'val_f1': 0.8201147485656429,
 'test_f1': 0.7476371345287519}

### Дистилляция слоёв

In [7]:
train(
    make_student(tokenizer, teacher_width=768),
    dataset,
    tokenizer,
    'layer_kd',
    **student_args
)

{'model': 'layer_kd',
 'params': 19971593,
 'val_f1': 0.8297161936560935,
 'test_f1': 0.7693387125065777}

### Wanda и SparseGPT

In [8]:
from hw3_distillation.prune import prune, sparsegpt, wanda

student_path = RUNS_DIR / 'student_t5' / 'best'
prune(
    student_path,
    dataset,
    tokenizer,
    wanda
)

{'model': 'wanda',
 'params': 19807753,
 'nonzero': 11943305,
 'val_f1': 0.8005690852791029,
 'test_f1': 0.729596333509607}

In [9]:
prune(
    student_path,
    dataset,
    tokenizer,
    sparsegpt
)

{'model': 'sparsegpt',
 'params': 19807753,
 'nonzero': 11943305,
 'val_f1': 0.8059251820236002,
 'test_f1': 0.7420294974829991}

### MiniLMv2

In [ ]:
short_run = dict(student_args, max_steps=4000)

train(
    make_student(tokenizer),
    dataset,
    tokenizer,
    'minilm_32_4k',
    relation_heads=32,
    **short_run
)

In [ ]:
train(
    make_student(tokenizer),
    dataset,
    tokenizer,
    'minilm_64_4k',
    relation_heads=64,
    **short_run
)

In [ ]:
train(
    make_student(tokenizer),
    dataset,
    tokenizer,
    'minilmv2',
    relation_heads=8,
    **student_args
)

In [10]:
train(
    make_student(tokenizer),
    dataset,
    tokenizer,
    'minilm_grid',
    relation_heads=32,
    **student_args
)

{'model': 'minilm_grid',
 'params': 19807753,
 'val_f1': 0.8356705013015366,
 'test_f1': 0.7638101983002833}

### Шеринг с LoRA

In [11]:
from hw3_distillation.models import add_lora

train(
    add_lora(RUNS_DIR / 'shared' / 'best', rank=8),
    dataset,
    tokenizer,
    'shared_lora',
    teacher_checkpoint=teacher_path,
    temperature=5.0,
    learning_rate=1e-3
)

{'model': 'shared_lora',
 'params': 7280137,
 'val_f1': 0.8199983138015344,
 'test_f1': 0.7464528069093151}

In [ ]:
train(
    add_lora(RUNS_DIR / 'shared' / 'best', rank=16),
    dataset,
    tokenizer,
    'lora_r16',
    teacher_checkpoint=teacher_path,
    temperature=5.0,
    learning_rate=1e-3
)

In [ ]:
train(
    add_lora(RUNS_DIR / 'shared' / 'best', rank=8),
    dataset,
    tokenizer,
    'lora_lr3',
    teacher_checkpoint=teacher_path,
    temperature=5.0,
    learning_rate=3e-4
)

### Инициализация прунингом

In [12]:
train(
    make_student(tokenizer),
    dataset,
    tokenizer,
    'student_4k',
    **short_run
)

{'model': 'student_4k',
 'params': 19807753,
 'val_f1': 0.7672914790865187,
 'test_f1': 0.6899345053429853}

In [13]:
from hw3_distillation.minitron import pruned_student

train(
    pruned_student(teacher_path, dataset, tokenizer, samples=1024),
    dataset,
    tokenizer,
    'pruned_1024',
    **short_run
)

{'model': 'pruned_1024',
 'params': 19807753,
 'val_f1': 0.8732673267326733,
 'test_f1': 0.8173027111034362}

In [ ]:
train(
    pruned_student(teacher_path, dataset, tokenizer, samples=256),
    dataset,
    tokenizer,
    'pruned_init',
    **short_run
)

In [ ]:
more_steps = dict(
    student_args,
    max_steps=2000,
    learning_rate=2e-5,
    eval_steps=500,
    save_steps=500
)

train(
    BertForNER.from_pretrained(RUNS_DIR / 'layer_kd' / 'best'),
    dataset,
    tokenizer,
    'layer_more',
    **more_steps
)

In [ ]:
train(
    BertForNER.from_pretrained(RUNS_DIR / 'pruned_1024' / 'best'),
    dataset,
    tokenizer,
    'pruned_more',
    **more_steps
)

некоторые раны конкретно в ноутбук уже в конце добавил, прогонял не в нем изначально части некоторые, но в wandb все есть